In [1]:
import json
import pickle 
from src.gpt_model import Model
import ast

DIMENSION = "Cohesion"
overall_rubric = '''
Native-like facility in the use of language with syntactic variety, Appropriate word choice and phrases; well-controlled text organization; precise use of grammar and conventions; rare language inaccuracies that do not impede communication.
'''

cohesion_rubric = "Text organization consistently well controlled using a variety of effective linguistic features such as reference and transitional words and phrases to connect ideas across sentences and paragraphs; appropriate overlap of ideas."

conventions_rubric = "Consistent use of appropriate conventions to convey meaning; spelling, capitalization, and punctuation errors nonexistent or negligible."

grammar_rubric = "Command of grammar and usage with few or no errors."

phraseology_rubric = "Flexible and effective use of a variety of phrases, such as idioms, collocations, and lexical bundles, to convey precise and subtle meanings; rare minor inaccuracies that are negligible."

syntax_rubric = "Flexible and effective use of a full range of syntactic structures including simple, compound, and complex sentences; There may be rare minor and negligible errors in sentence formation."

vocabulary_rubric = "Wide range of vocabulary flexibly and effectively used to convey precise meanings; skillful use of topic-related terms and less common words; rare negligible inaccuracies in word use."

DIMENSION_RUBRIC = {
    "Overall": overall_rubric,
    "Cohesion": cohesion_rubric, 
    "Conventions": conventions_rubric,
    "Grammar": grammar_rubric,
    "Phraseology": phraseology_rubric,
    "Syntax": syntax_rubric,
    "Vocabulary": vocabulary_rubric
}

with open(f"./data/ellipse/{DIMENSION}/human_llm_attributes.txt", "r") as file:
    attributes_list = file.read().splitlines()
attributes = "\n".join(attributes_list)

# LLM Initiation

In [2]:
with open("./api_keys.json", "r") as file:
    api_keys = json.load(file)

OPENAI_API_KEY = api_keys["openai"]

gpt4 = Model(model="gpt-4", temperature=0.0, api_key=OPENAI_API_KEY)

gpt-4


# Component Extraction

# Prompt Construction

In [3]:
with open(f"./prompts/ellipse/checklist_construction/component_extraction/system_prompt.txt", "r") as file:
    sys_prompt = file.read()

with open("./prompts/ellipse/checklist_construction/component_extraction/user_prompt.txt", "r") as file:
    user_prompt = file.read()

# Component Generation

In [4]:
prompt_list = [
    {"role":"system", "content": sys_prompt.format(DIMENSION, DIMENSION, DIMENSION_RUBRIC[DIMENSION])},
    {"role": "user", "content": user_prompt.format(attributes)}
]

gpt4_response = gpt4.ask_chatgpt(prompt_list)
components = ast.literal_eval(gpt4_response)

In [5]:
components

['Use of cohesive devices',
 'Effective overlap of ideas',
 'Clear presentation of ideas',
 'Control of text organization',
 'Variety of linguistic features used for cohesion']

# Attributes Clustering

In [6]:
with open("./prompts/ellipse/checklist_construction/attributes_clustering/system_prompt.txt", "r") as file:
    sys_prompt = file.read()

with open("./prompts/ellipse/checklist_construction/attributes_clustering/user_prompt.txt", "r") as file:
    user_prompt = file.read()

In [7]:
prompt_list = [
    {"role":"system", "content": sys_prompt},
    {"role": "user", "content": user_prompt.format(components, attributes)}
]

gpt4_response = gpt4.ask_chatgpt(prompt_list)
components_attributes_dic = eval(gpt4_response)

In [8]:
components_attributes = ""
for k, v in components_attributes_dic.items():
    components_attributes += f"{k}:\n{v}\n\n"
    

# Key Question Generation

In [9]:
with open("./prompts/ellipse/checklist_construction/question_generation/system_prompt.txt", "r") as file:
    sys_prompt = file.read()
with open("./prompts/ellipse/checklist_construction/question_generation/user_prompt.txt", "r") as file:
    user_prompt = file.read()

In [10]:
prompt_list = [
    {"role":"system", "content": sys_prompt.format(DIMENSION)},
    {"role": "user", "content": user_prompt.format(DIMENSION, DIMENSION, DIMENSION_RUBRIC[DIMENSION], components_attributes)}
]

generated_key_questions = gpt4.ask_chatgpt(prompt_list)

generated_key_questions = eval(generated_key_questions)

In [11]:
key_questions = ""
for component, question in generated_key_questions.items():
    key_questions += "- "+component+": "+question+"\n"

# Sub-question Generation

In [12]:
with open("./prompts/ellipse/checklist_construction/sub_question_generation/system_prompt.txt", "r") as file:
    sys_prompt = file.read()
with open("./prompts/ellipse/checklist_construction/sub_question_generation/user_prompt.txt", "r") as file:    
    user_prompt = file.read()

In [13]:
prompt_list = [
    {"role":"system", "content": sys_prompt},
    {"role": "user", "content": user_prompt.format(DIMENSION, DIMENSION, DIMENSION, DIMENSION_RUBRIC[DIMENSION], key_questions)}
]
generated_sub_questions = gpt4.ask_chatgpt(prompt_list)
generated_sub_questions = eval(generated_sub_questions)

In [14]:
sub_questions = ""
for component, sub_question_list in generated_sub_questions.items():
    for sub_question in sub_question_list:
        sub_questions += f"- {sub_question}\n"

# Question Validation

In [15]:
with open("./prompts/ellipse/checklist_construction/question_validation/system_prompt.txt", "r") as file:
    sys_prompt = file.read()
with open("./prompts/ellipse/checklist_construction/question_validation/user_prompt.txt", "r") as file:    
    user_prompt = file.read()

In [16]:
prompt_list = [
    {"role":"system", "content": sys_prompt},
    {"role": "user", "content": user_prompt.format(DIMENSION, DIMENSION, DIMENSION_RUBRIC[DIMENSION], DIMENSION, sub_questions)}
]

final_sub_questions = gpt4.ask_chatgpt(prompt_list)

final_sub_questions_list = ast.literal_eval(final_sub_questions)

In [17]:
checklist = ""

for sub_question in final_sub_questions_list:

    checklist+=f"- {sub_question}\n"

In [18]:
print(checklist)

- Does the essay incorporate transitional words and phrases to connect ideas?
- Are reference words used accurately to link sentences and paragraphs?
- Is there a diverse range of cohesive devices used throughout the essay?
- Does each paragraph naturally progress from the previous one?
- Does the essay maintain a consistent flow of ideas?
- Is the organization of ideas in the essay logical and easy to understand?
- Does the essay present its ideas in a manner that is easy to follow?
- Does the essay maintain a consistent structure throughout?
- Does the essay demonstrate a clear control over the organization of text?
- Does the essay use a variety of linguistic features to enhance cohesion?
- Does the essay avoid repetitive or mechanical usage of linguistic features?
- Are the linguistic features used effectively to enhance the overall cohesion of the essay?



In [19]:
with open(f"./data/ellipse/{DIMENSION}/{DIMENSION}_checklist.txt", "w") as file:
    file.write(checklist)